# KaushalDub: step recogniser training

Trains the model that finds procedure steps in a video, for three skills from the COIN
dataset (mobile screen protector, CPR, car polishing), plus five COIN cooking tasks used as
extra training data. One cooking task (omelette) is held out to test few-shot adaptation.

**Notebook settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**
(needs a phone-verified Kaggle account), Persistence **Files only**.

**Optional inputs (Add Input):**
- a private dataset named `kaushaldub-videos` holding videos you downloaded yourself,
  laid out as `<Task>/<youtube_id>.mp4` (use this if YouTube blocks downloads here);
- a private dataset named `yt-cookies` with a `cookies.txt` exported from your browser,
  which often gets past YouTube's bot check.

**Run order:** set `SMOKE = True`, Run All (about 3 minutes, synthetic data) to check the
setup. Then set `SMOKE = False` and Run All for the real run.

Data: COIN annotations (CC BY-NC 4.0), https://coin-dataset.github.io. Videos stay private.

In [ ]:
SMOKE = False           # True = synthetic features, checks the whole pipeline in minutes

TARGET = ["ReplaceMobileScreenProtector", "PerformCPR", "PolishCar"]
COOKING = ["CookOmelet", "MakePizza", "UseRiceCookerToCookRice", "BoilNoodles", "MakeCookie"]
HOLDOUT = "CookOmelet"      # never trained on; used for the few-shot test
FPS = 2                     # frames per second sampled from each video
MAX_FRAMES = 900            # 7.5 minutes cap per video
BACKBONES = ["clip", "resnet", "xclip"]
RUNS = [                    # (name, features, model)
    ("clip_tcn", "clip", "tcn"),            # main model
    ("resnet_bilstm", "resnet", "bilstm"),  # CNN-RNN baseline
    ("xclip_linear", "xclip", "linear"),    # pretrained video encoder baseline
    ("clip_bilstm", "clip", "bilstm"),      # ablation: same features, recurrent head
]
EPOCHS, LR, PATIENCE = 60, 1e-3, 12
MIN_LEN_S = 2.0             # smoothing: shortest step kept, in seconds
LOW_CONF = 0.6              # steps below this go to the LLM keyframe check
SEED = 7

import os
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
VIDEO_INPUTS = ["/kaggle/input/kaushaldub-videos", f"{WORK}/videos"]
COOKIES = "/kaggle/input/yt-cookies/cookies.txt"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
if SMOKE:
    EPOCHS, PATIENCE = 8, 8
print("work dir:", WORK, "| smoke test" if SMOKE else "| real run")

In [ ]:
!pip -q install -U open_clip_torch yt-dlp deno imageio-ffmpeg

In [ ]:
%%writefile kd_metrics.py
"""KaushalDub: decoding and evaluation helpers (numpy only, no torch).

Labels are integer arrays, one entry per sampled frame (FPS frames per second).
Class 0 is always "background" (no step happening)."""
import numpy as np

BG = 0


def softmax(logits, T=1.0):
    z = logits / T
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)


def to_segments(labels):
    """[0,0,2,2,2,1] -> [(0,0,2),(2,2,5),(1,5,6)] as (label, start, end) with end exclusive."""
    labels = np.asarray(labels)
    if len(labels) == 0:
        return []
    cut = np.flatnonzero(np.diff(labels)) + 1
    starts = np.r_[0, cut]
    ends = np.r_[cut, len(labels)]
    return [(int(labels[s]), int(s), int(e)) for s, e in zip(starts, ends)]


def smooth(labels, min_len):
    """Remove flicker: any segment shorter than min_len frames takes the label of its
    longer neighbour. Repeats until stable. Keeps the order of steps as they occur."""
    lab = np.asarray(labels).copy()
    if min_len <= 1 or len(lab) == 0:
        return lab
    for _ in range(len(lab)):
        segs = to_segments(lab)
        if len(segs) <= 1:
            break
        short = [i for i, (_, s, e) in enumerate(segs) if e - s < min_len]
        if not short:
            break
        # shortest first; on ties fix interior segments before the ones at the video edges
        i = min(short, key=lambda k: (segs[k][2] - segs[k][1], k == 0 or k == len(segs) - 1))
        prev = segs[i - 1] if i > 0 else None
        nxt = segs[i + 1] if i + 1 < len(segs) else None
        if prev is None:
            new = nxt[0]
        elif nxt is None:
            new = prev[0]
        else:
            new = prev[0] if (prev[2] - prev[1]) >= (nxt[2] - nxt[1]) else nxt[0]
        lab[segs[i][1]:segs[i][2]] = new
    return lab


def decode(probs, min_len):
    """probs [T,C] -> smoothed labels and per-segment confidence list."""
    lab = smooth(probs.argmax(-1), min_len)
    segs = []
    for c, s, e in to_segments(lab):
        segs.append({"label": c, "start": s, "end": e, "confidence": float(probs[s:e, c].mean())})
    return lab, segs


def macro_f1(pred, gt, n_classes):
    """Frame-level macro F1 over classes that occur in gt or pred."""
    pred, gt = np.asarray(pred), np.asarray(gt)
    f1s = []
    for c in range(n_classes):
        tp = np.sum((pred == c) & (gt == c))
        fp = np.sum((pred == c) & (gt != c))
        fn = np.sum((pred != c) & (gt == c))
        if tp + fp + fn == 0:
            continue
        f1s.append(2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1s)) if f1s else 0.0


def f1_at_iou(pred, gt, iou=0.5, bg=BG):
    """Segmental F1@IoU (MS-TCN convention), background segments ignored."""
    P = [s for s in to_segments(pred) if s[0] != bg]
    G = [s for s in to_segments(gt) if s[0] != bg]
    if not P and not G:
        return 1.0
    if not P or not G:
        return 0.0
    used = [False] * len(G)
    tp = 0
    for c, s, e in P:
        best, bj = 0.0, -1
        for j, (gc, gs, ge) in enumerate(G):
            if gc != c or used[j]:
                continue
            inter = max(0, min(e, ge) - max(s, gs))
            union = max(e, ge) - min(s, gs)
            if inter / union > best:
                best, bj = inter / union, j
        if best >= iou:
            tp += 1
            used[bj] = True
    fp = len(P) - tp
    fn = len(G) - tp
    return 2 * tp / (2 * tp + fp + fn)


def edit_score(pred, gt, bg=BG):
    """Order metric: 100 * (1 - normalised Levenshtein distance) between the sequences
    of step labels (background removed). 100 = same steps in the same order."""
    a = [s[0] for s in to_segments(pred) if s[0] != bg]
    b = [s[0] for s in to_segments(gt) if s[0] != bg]
    if not a and not b:
        return 100.0
    d = np.zeros((len(a) + 1, len(b) + 1))
    d[:, 0] = np.arange(len(a) + 1)
    d[0, :] = np.arange(len(b) + 1)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (a[i - 1] != b[j - 1]))
    return float(100 * (1 - d[-1, -1] / max(len(a), len(b))))


def boundary_f1(pred, gt, tol):
    """F1 of step boundaries: a predicted boundary is correct if within tol frames of an
    unmatched true boundary."""
    pb = np.flatnonzero(np.diff(np.asarray(pred))) + 1
    gb = np.flatnonzero(np.diff(np.asarray(gt))) + 1
    if len(pb) == 0 and len(gb) == 0:
        return 1.0
    if len(pb) == 0 or len(gb) == 0:
        return 0.0
    used = np.zeros(len(gb), bool)
    tp = 0
    for p in pb:
        d = np.abs(gb - p).astype(float)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= tol:
            used[j] = True
            tp += 1
    return 2 * tp / (len(pb) + len(gb))


def ece(probs, gt, bins=15):
    """Expected calibration error of the top-class confidence."""
    return ece_from(probs.max(-1), probs.argmax(-1) == gt, bins)


def ece_from(conf, correct, bins=15):
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(conf[m].mean() - correct[m].mean())
    return float(e)


def fit_temperature(logits, gt):
    """Single temperature T that minimises NLL on validation frames (temperature scaling)."""
    from scipy.optimize import minimize_scalar
    gt = np.asarray(gt)

    def nll(logT):
        p = softmax(logits, np.exp(logT))
        return -np.mean(np.log(p[np.arange(len(gt)), gt] + 1e-12))

    r = minimize_scalar(nll, bounds=(np.log(0.05), np.log(20)), method="bounded")
    return float(np.exp(r.x))


def evaluate(videos, fps, min_len_s=2.0, tol_s=2.0):
    """videos: list of dicts with 'probs' [T,C], 'gt' [T], 'n_classes'.
    Returns averaged metrics (percent)."""
    ml = max(1, int(round(min_len_s * fps)))
    tol = tol_s * fps
    rows = []
    allp, allg, allpr = [], [], []
    for v in videos:
        lab, _ = decode(v["probs"], ml)
        g = v["gt"]
        rows.append({
            "acc": float(np.mean(lab == g)),
            "f1@50": f1_at_iou(lab, g, 0.5),
            "edit": edit_score(lab, g) / 100,
            "bf1@2s": boundary_f1(lab, g, tol),
        })
        allp.append(lab); allg.append(g)
        allpr.append((v["probs"].max(-1), v["probs"].argmax(-1) == g))
    out = {k: 100 * float(np.mean([r[k] for r in rows])) for k in rows[0]} if rows else {}
    if rows:
        n = videos[0]["n_classes"]
        out["macro_f1"] = 100 * macro_f1(np.concatenate(allp), np.concatenate(allg), n)
        out["ece"] = 100 * ece_from(np.concatenate([c for c, _ in allpr]), np.concatenate([k for _, k in allpr]))
        out["videos"] = len(rows)
    return out

In [ ]:
%%writefile kd_model.py
"""KaushalDub step recogniser: models, frame reading, feature encoders and inference.

Main model: frozen CLIP ViT-B/16 frame features -> multi-stage temporal conv net (MS-TCN
style) -> cosine heads, one per task, initialised from CLIP text embeddings of the step
names. Baselines: ResNet-50 features + BiLSTM, X-CLIP features + linear head."""
import json, math, os, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kd_metrics import softmax, decode

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
IMNET_MEAN = (0.485, 0.456, 0.406)
IMNET_STD = (0.229, 0.224, 0.225)
PROMPTS = ["{}", "a video of a person who is about to {}", "how to {}: a step in a tutorial video"]
BG_PROMPTS = ["a person talking to the camera", "the title screen or introduction of a tutorial video",
              "a finished result being shown at the end of a video"]


# ---------------------------------------------------------------- frames
def ffmpeg_exe():
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return "ffmpeg"


def read_frames(path, fps=2, size=224, max_frames=900):
    """Decode a video at `fps` frames per second, resized and centre-cropped to size x size.
    Returns uint8 array [T, size, size, 3]."""
    vf = f"fps={fps},scale={size}:{size}:force_original_aspect_ratio=increase,crop={size}:{size}"
    cmd = [ffmpeg_exe(), "-loglevel", "error", "-i", path, "-vf", vf, "-frames:v", str(max_frames),
           "-f", "rawvideo", "-pix_fmt", "rgb24", "-"]
    out = subprocess.run(cmd, capture_output=True, check=False).stdout
    return np.frombuffer(out, np.uint8).reshape(-1, size, size, 3)


def _to_tensor(frames, mean, std, device):
    x = torch.from_numpy(np.array(frames, copy=True)).to(device).permute(0, 3, 1, 2).float() / 255
    m = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    s = torch.tensor(std, device=device).view(1, 3, 1, 1)
    return (x - m) / s


# ---------------------------------------------------------------- encoders
class ClipEncoder:
    name, dim = "clip", 512

    def __init__(self, device):
        import open_clip
        self.model, _, _ = open_clip.create_model_and_transforms("ViT-B-16-quickgelu", pretrained="openai")
        self.model = self.model.to(device).eval().half() if device != "cpu" else self.model.eval()
        self.tok = open_clip.get_tokenizer("ViT-B-16-quickgelu")
        self.device = device

    @torch.no_grad()
    def frames(self, frames, bs=128):
        out = []
        for i in range(0, len(frames), bs):
            x = _to_tensor(frames[i:i + bs], CLIP_MEAN, CLIP_STD, self.device)
            if self.device != "cpu":
                x = x.half()
            out.append(F.normalize(self.model.encode_image(x).float(), dim=-1).cpu())
        return torch.cat(out).numpy() if out else np.zeros((0, self.dim), np.float32)

    @torch.no_grad()
    def text(self, sentences):
        t = self.tok(sentences).to(self.device)
        return F.normalize(self.model.encode_text(t).float(), dim=-1).cpu().numpy()

    def step_embeddings(self, steps):
        """steps: ['background', 'wipe the screen', ...] -> [C, 512] averaged over prompts."""
        rows = []
        for s in steps:
            s = s.replace("in the mouse", "in the mouth")      # typo in the COIN label
            sents = BG_PROMPTS if s == "background" else [p.format(s) for p in PROMPTS]
            e = self.text(sents).mean(0)
            rows.append(e / np.linalg.norm(e))
        return np.stack(rows).astype(np.float32)


class ResNetEncoder:
    name, dim = "resnet", 2048

    def __init__(self, device):
        import torchvision
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Identity()
        self.model, self.device = m.to(device).eval(), device

    @torch.no_grad()
    def frames(self, frames, bs=128):
        out = []
        for i in range(0, len(frames), bs):
            x = _to_tensor(frames[i:i + bs], IMNET_MEAN, IMNET_STD, self.device)
            with torch.autocast("cuda", enabled=self.device != "cpu"):
                out.append(self.model(x).float().cpu())
        return torch.cat(out).numpy() if out else np.zeros((0, self.dim), np.float32)


class XClipEncoder:
    """Video encoder: 8-frame windows (4 s at 2 fps), one window per second; each frame gets
    the embedding of the window centred on it."""
    name, dim = "xclip", 512

    def __init__(self, device):
        from transformers import XCLIPModel
        self.model = XCLIPModel.from_pretrained("microsoft/xclip-base-patch32").to(device).eval()
        self.device = device

    @torch.no_grad()
    def frames(self, frames, win=8, stride=2, bs=8):
        T = len(frames)
        if T == 0:
            return np.zeros((0, self.dim), np.float32)
        centres = list(range(0, T, stride))
        idx = [np.clip(np.arange(c - win // 2, c + win // 2), 0, T - 1) for c in centres]
        embs = []
        for i in range(0, len(idx), bs):
            clips = np.stack([frames[j] for j in idx[i:i + bs]])          # [B,8,H,W,3]
            B = clips.shape[0]
            x = _to_tensor(clips.reshape(-1, *clips.shape[2:]), CLIP_MEAN, CLIP_STD, self.device)
            x = x.view(B, win, 3, x.shape[-2], x.shape[-1])
            with torch.autocast("cuda", enabled=self.device != "cpu"):
                e = self.model.get_video_features(pixel_values=x)
            if not torch.is_tensor(e):          # newer transformers return an output object
                e = e.pooler_output if getattr(e, "pooler_output", None) is not None else e[0]
            if e.dim() == 3:                    # [B, tokens, D] -> [B, D]
                e = e.mean(1)
            embs.append(F.normalize(e.float(), dim=-1).cpu())
        E = torch.cat(embs).numpy()
        return np.repeat(E, stride, axis=0)[:T]


ENCODERS = {"clip": ClipEncoder, "resnet": ResNetEncoder, "xclip": XClipEncoder}


# ---------------------------------------------------------------- models
class DilatedResidual(nn.Module):
    def __init__(self, dilation, ch, p=0.3):
        super().__init__()
        self.c1 = nn.Conv1d(ch, ch, 3, padding=dilation, dilation=dilation)
        self.c2 = nn.Conv1d(ch, ch, 1)
        self.dp = nn.Dropout(p)

    def forward(self, x):
        return x + self.dp(self.c2(F.relu(self.c1(x))))


class TCNStage(nn.Module):
    def __init__(self, din, ch, layers):
        super().__init__()
        self.inp = nn.Conv1d(din, ch, 1)
        self.layers = nn.ModuleList([DilatedResidual(2 ** i, ch) for i in range(layers)])

    def forward(self, x):                      # x [B, D, T]
        x = self.inp(x)
        for l in self.layers:
            x = l(x)
        return x


class CosineHeads(nn.Module):
    """One weight matrix per task; logits = scale * cos(embedding, class vector)."""

    def __init__(self, emb, ncls, init=None):
        super().__init__()
        self.W = nn.ParameterDict()
        for t, n in ncls.items():
            w = torch.as_tensor(init[t]).float().clone() if init and t in init else torch.randn(n, emb) * 0.02
            self.W[t] = nn.Parameter(w)
        self.logscale = nn.Parameter(torch.tensor(math.log(15.0)))

    def add_task(self, task, weights):
        self.W[task] = nn.Parameter(torch.as_tensor(weights).float().clone().to(self.logscale.device))

    def forward(self, z, task):
        return self.logscale.exp() * z @ F.normalize(self.W[task], dim=-1).t()


class StepTCN(nn.Module):
    """Main model. When the input is CLIP (din == emb) the embedding starts as the CLIP frame
    feature itself (residual, zero-initialised projection), so text-initialised heads give
    zero-shot predictions before any training."""

    def __init__(self, din, ncls, emb=512, ch=128, layers=6, stages=2, text_init=None):
        super().__init__()
        self.cfg = dict(din=din, ncls=ncls, emb=emb, ch=ch, layers=layers, stages=stages)
        self.residual = din == emb
        self.drop = nn.Dropout(0.2)
        self.stages = nn.ModuleList([TCNStage(din, ch, layers)] +
                                    [TCNStage(ch, ch, layers) for _ in range(stages - 1)])
        self.proj = nn.ModuleList([nn.Conv1d(ch, emb, 1) for _ in range(stages)])
        if self.residual:
            for p in self.proj:
                nn.init.zeros_(p.weight); nn.init.zeros_(p.bias)
        self.heads = CosineHeads(emb, ncls, text_init)

    def embed_all(self, x):                    # x [B, T, D] -> list of [B, T, E]
        base = x.transpose(1, 2)
        h = self.drop(base)
        zs = []
        for st, pr in zip(self.stages, self.proj):
            h = st(h)
            z = pr(h) + (base if self.residual else 0)
            zs.append(F.normalize(z.transpose(1, 2), dim=-1))
        return zs

    def embed(self, x):
        return self.embed_all(x)[-1]

    def forward(self, x, task):
        return [self.heads(z, task) for z in self.embed_all(x)]


class BiLSTMNet(nn.Module):
    """Baseline (professor's CNN-RNN): frame features -> 2-layer BiLSTM -> linear head per task."""

    def __init__(self, din, ncls, hid=256):
        super().__init__()
        self.cfg = dict(din=din, ncls=ncls, hid=hid)
        self.drop = nn.Dropout(0.2)
        self.lstm = nn.LSTM(din, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.heads = nn.ModuleDict({t: nn.Linear(2 * hid, n) for t, n in ncls.items()})

    def forward(self, x, task):
        h, _ = self.lstm(self.drop(x))
        return [self.heads[task](h)]


class LinearNet(nn.Module):
    """Baseline for video-encoder features: per-frame linear head per task."""

    def __init__(self, din, ncls):
        super().__init__()
        self.cfg = dict(din=din, ncls=ncls)
        self.drop = nn.Dropout(0.2)
        self.heads = nn.ModuleDict({t: nn.Linear(din, n) for t, n in ncls.items()})

    def forward(self, x, task):
        return [self.heads[task](self.drop(x))]


def build_model(kind, din, ncls, text_init=None):
    if kind == "tcn":
        return StepTCN(din, ncls, text_init=text_init)
    if kind == "bilstm":
        return BiLSTMNet(din, ncls)
    if kind == "linear":
        return LinearNet(din, ncls)
    raise ValueError(kind)


# ---------------------------------------------------------------- inference (used by the backend)
class StepRecognizer:
    """Load an exported bundle and turn a video into timed steps.

    rec = StepRecognizer('kaushaldub_model'); rec.predict('video.mp4', 'PerformCPR')"""

    def __init__(self, bundle_dir, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        cfg = json.load(open(os.path.join(bundle_dir, "config.json")))
        self.cfg = cfg
        m = cfg["model"]
        self.model = StepTCN(m["din"], {t: len(v) for t, v in cfg["vocab"].items()}, m["emb"], m["ch"],
                             m["layers"], m["stages"])
        self.model.load_state_dict(torch.load(os.path.join(bundle_dir, "model.pt"), map_location="cpu"))
        self.model.to(self.device).eval()
        self._enc = None

    @property
    def encoder(self):
        if self._enc is None:
            self._enc = ClipEncoder(self.device)
        return self._enc

    def tasks(self):
        return list(self.cfg["vocab"].keys())

    @torch.no_grad()
    def predict_features(self, feats, task):
        x = torch.from_numpy(np.asarray(feats, np.float32))[None].to(self.device)
        logits = self.model(x, task)[-1][0].float().cpu().numpy()
        return softmax(logits, self.cfg["temperature"].get(task, 1.0))

    def predict(self, video_path, task):
        fps = self.cfg["fps"]
        frames = read_frames(video_path, fps, 224, self.cfg["max_frames"])
        probs = self.predict_features(self.encoder.frames(frames), task)
        return self.format(probs, task)

    def format(self, probs, task):
        fps, vocab = self.cfg["fps"], self.cfg["vocab"][task]
        _, segs = decode(probs, max(1, int(round(self.cfg["min_len_s"] * fps))))
        steps = []
        for s in segs:
            if s["label"] == 0:
                continue
            steps.append({"idx": len(steps) + 1, "label": vocab[s["label"]],
                          "start": round(s["start"] / fps, 2), "end": round(s["end"] / fps, 2),
                          "confidence": round(s["confidence"], 3),
                          "low_confidence": s["confidence"] < self.cfg["low_conf"],
                          "keyframe_s": round((s["start"] + s["end"]) / 2 / fps, 2)})
        return {"task": task, "duration_s": round(len(probs) / fps, 2), "steps": steps}

In [ ]:
import json, glob, random, time, shutil, subprocess, collections, copy
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from concurrent.futures import ThreadPoolExecutor

os.chdir(WORK)
import sys; sys.path.insert(0, WORK)
from kd_metrics import *
from kd_model import *

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "| GPUs:", torch.cuda.device_count())

## 1. Manifest: which videos, which steps, which split

In [ ]:
if not os.path.exists("annotations/COIN.json"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/coin-dataset/annotations.git"], check=True)
db = json.load(open("annotations/COIN.json"))["database"]

TASKS = TARGET + COOKING
VOCAB, VIDEOS = {}, []
rng = random.Random(SEED)
for t in TASKS:
    vs = {k: v for k, v in db.items() if v["class"] == t}
    cnt = collections.Counter(a["label"] for v in vs.values() for a in v["annotation"])
    VOCAB[t] = ["background"] + [s for s, _ in cnt.most_common()]
    train_ids = sorted(k for k, v in vs.items() if v["subset"] == "training")
    rng.shuffle(train_ids)
    val = set(train_ids[: max(3, round(0.15 * len(train_ids)))])
    for k, v in vs.items():
        split = "test" if v["subset"] == "testing" else ("val" if k in val else "train")
        segs = sorted(({"label": a["label"], "start": a["segment"][0], "end": a["segment"][1]}
                       for a in v["annotation"]), key=lambda s: s["start"])
        VIDEOS.append({"id": k, "task": t, "split": split, "duration": v["duration"],
                       "url": "https://www.youtube.com/watch?v=" + k, "segments": segs})
json.dump({"vocab": VOCAB, "videos": VIDEOS}, open("manifest.json", "w"), indent=1)
tab = collections.Counter((v["task"], v["split"]) for v in VIDEOS)
for t in TASKS:
    print(f"{t:30s} train {tab[(t,'train')]:3d} val {tab[(t,'val')]:3d} test {tab[(t,'test')]:3d} | steps: {VOCAB[t][1:]}")

def frame_labels(v, T):
    """Per-frame class index (0 = background) for a video sampled at FPS."""
    y = np.zeros(T, np.int64)
    t = (np.arange(T) + 0.5) / FPS
    for s in v["segments"]:
        y[(t >= s["start"]) & (t < s["end"])] = VOCAB[v["task"]].index(s["label"])
    return y

## 2. Videos: use uploaded copies, download the rest

In [ ]:
def find_video(v):
    for root in VIDEO_INPUTS:
        for f in glob.glob(f"{root}/**/{v['id']}.*", recursive=True):
            if f.rsplit(".", 1)[-1].lower() in ("mp4", "webm", "mkv", "mov"):
                return f
    return None

def download(v):
    if find_video(v):
        return True
    out = f"{WORK}/videos/{v['task']}/{v['id']}.%(ext)s"
    # video-only stream is enough for training (no audio needed); deno + ejs solve YouTube's JS challenge
    cmd = ["yt-dlp", "-q", "--no-warnings", "--retries", "2", "--remote-components", "ejs:github",
           "-f", "bv*[height<=360]/b[height<=360]/bv*/b", "-o", out, v["url"]]
    if os.path.exists(COOKIES):
        cmd[1:1] = ["--cookies", COOKIES]
    subprocess.run(cmd, capture_output=True, timeout=600)
    return find_video(v) is not None

if not SMOKE:
    t0 = time.time()
    with ThreadPoolExecutor(6) as ex:
        ok = list(ex.map(download, VIDEOS))
    have = collections.Counter(v["task"] for v, o in zip(VIDEOS, ok) if o)
    for t in TASKS:
        n = sum(v["task"] == t for v in VIDEOS)
        print(f"{t:30s} {have[t]:3d}/{n} videos available")
    print(f"done in {(time.time()-t0)/60:.1f} min")
    if sum(ok) < 0.5 * len(VIDEOS):
        print("\nWARNING: most downloads failed (YouTube bot check is likely). Add a `yt-cookies` input with "
              "cookies.txt, or download on your own computer with download_videos.py and upload the folder "
              "as the `kaushaldub-videos` dataset. Then rerun from this cell.")

## 3. Frame features (one-time, cached as .npy)

In [ ]:
def feat_path(bk, vid):
    return f"{WORK}/feats/{bk}/{vid}.npy"

def make_smoke_features():
    """Synthetic stand-in: each (task, step) has a direction in feature space; frames are that
    direction plus noise. Lets the whole pipeline run without videos or GPUs."""
    dims = {"clip": 512, "resnet": 2048, "xclip": 512}
    g = np.random.default_rng(SEED)
    centres = {bk: {t: g.normal(size=(len(VOCAB[t]), d)) for t in TASKS} for bk, d in dims.items()}
    per_task = collections.Counter()
    for v in VIDEOS:
        key = (v["task"], v["split"])
        if per_task[key] >= {"train": 20, "val": 5, "test": 8}[v["split"]]:
            continue
        per_task[key] += 1
        T = int(min(MAX_FRAMES, v["duration"] * FPS))
        y = frame_labels(v, T)
        np.save(f"{WORK}/labels/{v['id']}.npy", y.astype(np.int16))
        for bk, d in dims.items():
            x = centres[bk][v["task"]][y] + 2.0 * g.normal(size=(T, d))
            x /= np.linalg.norm(x, axis=1, keepdims=True)
            os.makedirs(f"{WORK}/feats/{bk}", exist_ok=True)
            np.save(feat_path(bk, v["id"]), x.astype(np.float16))
    return centres

# never mix synthetic and real features: a marker file records which kind is on disk
if os.path.exists(f"{WORK}/feats/SMOKE_DATA") != SMOKE:
    shutil.rmtree(f"{WORK}/feats", ignore_errors=True)
    shutil.rmtree(f"{WORK}/labels", ignore_errors=True)
os.makedirs(f"{WORK}/labels", exist_ok=True)
os.makedirs(f"{WORK}/feats", exist_ok=True)
if SMOKE:
    shutil.rmtree(f"{WORK}/feats", ignore_errors=True)
    SMOKE_CENTRES = make_smoke_features()
    open(f"{WORK}/feats/SMOKE_DATA", "w").write("synthetic")
else:
    encs = {}
    for bk in BACKBONES:
        try:
            encs[bk] = ENCODERS[bk](DEV)
        except Exception as e:
            print(f"skipping {bk}: {e}")
    todo = [v for v in VIDEOS if find_video(v)
            and not all(os.path.exists(feat_path(bk, v["id"])) for bk in encs)]
    print(f"extracting features for {len(todo)} videos with {list(encs)}")
    t0 = time.time()
    for i, v in enumerate(todo):
        try:
            frames = read_frames(find_video(v), FPS, 224, MAX_FRAMES)
        except Exception as e:
            print("read failed", v["id"], e); continue
        if len(frames) < 4:
            print("too short / unreadable:", v["id"]); continue
        np.save(f"{WORK}/labels/{v['id']}.npy", frame_labels(v, len(frames)).astype(np.int16))
        for bk, enc in encs.items():
            if os.path.exists(feat_path(bk, v["id"])):
                continue
            os.makedirs(f"{WORK}/feats/{bk}", exist_ok=True)
            try:
                np.save(feat_path(bk, v["id"]), enc.frames(frames).astype(np.float16))
            except Exception as e:
                print(f"{bk} failed on {v['id']}: {e}")
        if i % 25 == 0:
            print(f"{i+1}/{len(todo)} videos, {(time.time()-t0)/60:.1f} min")
    print("features done")

# CLIP text embeddings of every step name: used to initialise the heads and for zero-shot
if SMOKE:
    g = np.random.default_rng(1)
    TEXT = {t: (c / np.linalg.norm(c, axis=1, keepdims=True) + 0.05 * g.normal(size=c.shape)).astype(np.float32)
            for t, c in SMOKE_CENTRES["clip"].items()}
    for t in TEXT:
        TEXT[t] /= np.linalg.norm(TEXT[t], axis=1, keepdims=True)
else:
    clip_enc = encs.get("clip") or ClipEncoder(DEV)
    TEXT = {t: clip_enc.step_embeddings(VOCAB[t]) for t in TASKS}
np.save("text_embeddings.npy", TEXT, allow_pickle=True)

## 4. Training

In [ ]:
def load_items(bk, split, tasks, limit=None, seed=SEED):
    """Cached features + labels for one split. limit = max videos per task (data-efficiency runs)."""
    items, per = [], collections.Counter()
    vids = [v for v in VIDEOS if v["split"] == split and v["task"] in tasks]
    random.Random(seed).shuffle(vids)
    for v in vids:
        f = feat_path(bk, v["id"])
        if not os.path.exists(f):
            continue
        if limit and per[v["task"]] >= limit:
            continue
        x = np.load(f).astype(np.float32)
        y = np.load(f"{WORK}/labels/{v['id']}.npy").astype(np.int64)
        n = min(len(x), len(y))
        items.append({"id": v["id"], "task": v["task"], "x": torch.from_numpy(x[:n]), "y": torch.from_numpy(y[:n])})
        per[v["task"]] += 1
    return items

def class_weights(items):
    w = {}
    for t in {i["task"] for i in items}:
        n = len(VOCAB[t])
        c = np.bincount(np.concatenate([i["y"].numpy() for i in items if i["task"] == t]), minlength=n) + 1
        wt = c ** -0.5
        w[t] = torch.tensor(wt / wt.mean(), dtype=torch.float32, device=DEV)
    return w

def smoothing_loss(logits):
    lp = F.log_softmax(logits, -1)
    return torch.clamp((lp[:, 1:] - lp[:, :-1].detach()) ** 2, max=16).mean()

@torch.no_grad()
def predict_logits(model, items):
    model.eval()
    return [model(i["x"][None].to(DEV), i["task"])[-1][0].float().cpu().numpy() for i in items]

def evaluate_items(model, items, temps=None):
    """Per-task metrics. temps: task -> temperature (1.0 if missing)."""
    out = {}
    logits = predict_logits(model, items)
    for t in sorted({i["task"] for i in items}):
        vids = [{"probs": softmax(l, (temps or {}).get(t, 1.0)), "gt": i["y"].numpy(), "n_classes": len(VOCAB[t])}
                for l, i in zip(logits, items) if i["task"] == t]
        out[t] = evaluate(vids, FPS, MIN_LEN_S)
    return out

def fit_temps(model, items):
    logits = predict_logits(model, items)
    temps = {}
    for t in sorted({i["task"] for i in items}):
        L = np.concatenate([l for l, i in zip(logits, items) if i["task"] == t])
        Y = np.concatenate([i["y"].numpy() for i in items if i["task"] == t])
        temps[t] = fit_temperature(L, Y)
    return temps

def mean_over(res, tasks, key):
    vals = [res[t][key] for t in tasks if t in res]
    return float(np.mean(vals)) if vals else float("nan")

def train(model, tr, va, epochs=EPOCHS, lr=LR, patience=PATIENCE, params=None, log=True):
    model.to(DEV)
    opt = torch.optim.AdamW(params if params is not None else model.parameters(), lr=lr, weight_decay=1e-4)
    cw = class_weights(tr)
    best, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        random.shuffle(tr)
        tot = 0.0
        for it in tr:
            x, y = it["x"][None].to(DEV), it["y"].to(DEV)
            outs = model(x, it["task"])
            loss = sum(F.cross_entropy(o[0], y, weight=cw[it["task"]]) + 0.15 * smoothing_loss(o) for o in outs)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); tot += loss.item()
        res = evaluate_items(model, va)
        score = float(np.mean([r["macro_f1"] for r in res.values()]))
        if score > best:
            best, best_state, bad = score, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
        if log and (ep % 5 == 0 or bad == 0):
            print(f"  epoch {ep:3d} loss {tot/len(tr):.3f} val macro-F1 {score:.1f}" + (" *" if bad == 0 else ""))
        if bad >= patience:
            break
    model.load_state_dict(best_state)
    return model, best

TRAIN_TASKS = [t for t in TASKS if t != HOLDOUT]
RESULTS, MODELS = {}, {}
for name, bk, kind in RUNS:
    tr = load_items(bk, "train", TRAIN_TASKS)
    if not tr:
        print(f"{name}: no {bk} features, skipped"); continue
    va, te = load_items(bk, "val", TRAIN_TASKS), load_items(bk, "test", TRAIN_TASKS)
    din = tr[0]["x"].shape[1]
    ncls = {t: len(VOCAB[t]) for t in TRAIN_TASKS}
    text_init = {t: TEXT[t] for t in TRAIN_TASKS} if (kind == "tcn" and bk == "clip") else None
    model = build_model(kind, din, ncls, text_init)
    print(f"\n== {name}: {len(tr)} train / {len(va)} val / {len(te)} test videos, "
          f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M trainable params")
    t0 = time.time()
    model, _ = train(model, tr, va)
    temps = fit_temps(model, va)
    raw = evaluate_items(model, te)
    cal = evaluate_items(model, te, temps)
    for t in cal:
        cal[t]["ece_raw"] = raw[t]["ece"]
    RESULTS[name] = {"per_task": cal, "temps": temps, "minutes": (time.time() - t0) / 60}
    MODELS[name] = model.cpu()
    print(f"   target macro-F1 {mean_over(cal, TARGET, 'macro_f1'):.1f} | F1@50 {mean_over(cal, TARGET, 'f1@50'):.1f}"
          f" | edit {mean_over(cal, TARGET, 'edit'):.1f} | ECE {mean_over(cal, TARGET, 'ece_raw'):.1f} -> {mean_over(cal, TARGET, 'ece'):.1f}")

## 5. Zero-shot and data-efficiency (how many videos does a skill need?)

In [ ]:
class ZeroShot(nn.Module):
    """No training at all: CLIP frame features vs CLIP text embeddings of the step names."""
    def __init__(self):
        super().__init__()
        self.scale = 50.0
    def forward(self, x, task):
        return [self.scale * x @ torch.as_tensor(TEXT[task]).to(x.device).t()]

te_clip = load_items("clip", "test", TARGET)
if te_clip:
    zs = ZeroShot().to(DEV)
    RESULTS["clip_zero_shot"] = {"per_task": evaluate_items(zs, te_clip), "temps": {}, "minutes": 0}

CURVE = {}
for n in ([5, 10] if SMOKE else [5, 10, 20, None]):
    tr = load_items("clip", "train", TARGET, limit=n) + load_items("clip", "train", [t for t in TRAIN_TASKS if t not in TARGET])
    va = load_items("clip", "val", TRAIN_TASKS)
    if not tr:
        break
    m = build_model("tcn", tr[0]["x"].shape[1], {t: len(VOCAB[t]) for t in TRAIN_TASKS}, {t: TEXT[t] for t in TRAIN_TASKS})
    m, _ = train(m, tr, va, log=False)
    r = evaluate_items(m, te_clip)
    CURVE[str(n or "all")] = {"macro_f1": mean_over(r, TARGET, "macro_f1"), "f1@50": mean_over(r, TARGET, "f1@50")}
    print(f"{str(n or 'all'):>4s} videos per skill -> macro-F1 {CURVE[str(n or 'all')]['macro_f1']:.1f}, F1@50 {CURVE[str(n or 'all')]['f1@50']:.1f}")

## 6. Few-shot: adapt to an unseen task (omelette) from K labelled videos

In [ ]:
def prototypes(model, support, task, alpha=0.5):
    """Class vector = mix of the CLIP text embedding and the mean embedding of support frames."""
    text = torch.as_tensor(TEXT[task])
    if not support:
        return text
    model.eval()
    with torch.no_grad():
        Z = torch.cat([model.embed(s["x"][None].to(DEV))[0].cpu() for s in support])
    Y = torch.cat([s["y"] for s in support])
    P = text.clone()
    for c in range(len(VOCAB[task])):
        if (Y == c).any():
            P[c] = F.normalize(alpha * text[c] + (1 - alpha) * F.normalize(Z[Y == c].mean(0), dim=0), dim=0)
    return P

FEW = {}
base = MODELS.get("clip_tcn")
sup_pool = load_items("clip", "train", [HOLDOUT]) + load_items("clip", "val", [HOLDOUT])
te_h = load_items("clip", "test", [HOLDOUT])
if base is not None and te_h:
    base = base.to(DEV)
    for K in [0, 1, 5, 10]:
        scores = {"prototype": [], "finetune": []}
        for draw in range(1 if K == 0 else 3):
            sup = random.Random(draw).sample(sup_pool, min(K, len(sup_pool)))
            m = copy.deepcopy(base)
            m.heads.add_task(HOLDOUT, prototypes(m, sup, HOLDOUT))
            scores["prototype"].append(evaluate_items(m, te_h)[HOLDOUT])
            if K >= 5:   # also fine-tune only the new head on the support videos
                m2 = copy.deepcopy(m)
                for p in m2.parameters():
                    p.requires_grad = False
                m2.heads.W[HOLDOUT].requires_grad = True
                m2, _ = train(m2, list(sup), list(sup), epochs=30, lr=5e-3, patience=30,
                              params=[m2.heads.W[HOLDOUT]], log=False)
                scores["finetune"].append(evaluate_items(m2, te_h)[HOLDOUT])
        for meth, rs in scores.items():
            if rs:
                FEW[f"{meth}_K{K}"] = {k: float(np.mean([r[k] for r in rs])) for k in ["macro_f1", "f1@50", "edit", "acc"]}
                print(f"K={K:2d} {meth:9s} macro-F1 {FEW[f'{meth}_K{K}']['macro_f1']:.1f}  F1@50 {FEW[f'{meth}_K{K}']['f1@50']:.1f}")
    # keep a 10-shot omelette head in the exported model
    HOLDOUT_HEAD = prototypes(base, random.Random(0).sample(sup_pool, min(10, len(sup_pool))), HOLDOUT)
    base.cpu()

## 7. Results table

In [ ]:
def fmt(x):
    return "–" if x != x else f"{x:.1f}"

lines = ["| Model | Target macro-F1 | F1@50 | Edit (order) | Boundary F1 @2s | ECE raw → calibrated |",
         "|---|---|---|---|---|---|"]
for name, r in RESULTS.items():
    p = r["per_task"]
    lines.append(f"| {name} | {fmt(mean_over(p, TARGET, 'macro_f1'))} | {fmt(mean_over(p, TARGET, 'f1@50'))} | "
                 f"{fmt(mean_over(p, TARGET, 'edit'))} | {fmt(mean_over(p, TARGET, 'bf1@2s'))} | "
                 f"{fmt(mean_over(p, TARGET, 'ece_raw') if 'ece_raw' in next(iter(p.values())) else float('nan'))} → {fmt(mean_over(p, TARGET, 'ece'))} |")
lines += ["", "Per skill (main model, clip_tcn):", "", "| Skill | macro-F1 | F1@50 | Edit | Accuracy | Test videos |", "|---|---|---|---|---|---|"]
if "clip_tcn" in RESULTS:
    for t in TARGET + [c for c in COOKING if c != HOLDOUT]:
        r = RESULTS["clip_tcn"]["per_task"].get(t)
        if r:
            lines.append(f"| {t} | {fmt(r['macro_f1'])} | {fmt(r['f1@50'])} | {fmt(r['edit'])} | {fmt(r['acc'])} | {r['videos']} |")
if CURVE:
    lines += ["", "Videos per skill vs score (clip_tcn):", "", "| Videos per skill | macro-F1 | F1@50 |", "|---|---|---|"]
    lines += [f"| {k} | {fmt(v['macro_f1'])} | {fmt(v['f1@50'])} |" for k, v in CURVE.items()]
if FEW:
    lines += ["", f"Few-shot on unseen task ({HOLDOUT}):", "", "| Method | macro-F1 | F1@50 | Edit |", "|---|---|---|---|"]
    lines += [f"| {k} | {fmt(v['macro_f1'])} | {fmt(v['f1@50'])} | {fmt(v['edit'])} |" for k, v in FEW.items()]
REPORT = "\n".join(lines)
print(REPORT)
open("results.md", "w").write(REPORT + "\n")
json.dump({"runs": RESULTS, "curve": CURVE, "few_shot": FEW, "smoke": SMOKE}, open("results.json", "w"), indent=1, default=float)

## 8. Export the model bundle for the app

In [ ]:
assert "clip_tcn" in MODELS, "main model was not trained"
best = copy.deepcopy(MODELS["clip_tcn"])
temps = dict(RESULTS["clip_tcn"]["temps"])
vocab = {t: VOCAB[t] for t in TRAIN_TASKS}
if "HOLDOUT_HEAD" in globals():
    best.heads.add_task(HOLDOUT, HOLDOUT_HEAD.cpu())
    vocab[HOLDOUT] = VOCAB[HOLDOUT]
BUNDLE = f"{WORK}/kaushaldub_model"
shutil.rmtree(BUNDLE, ignore_errors=True); os.makedirs(BUNDLE)
torch.save(best.state_dict(), f"{BUNDLE}/model.pt")
cfg = {"backbone": {"name": "ViT-B-16-quickgelu", "pretrained": "openai", "library": "open_clip"},
       "model": {k: v for k, v in best.cfg.items() if k != "ncls"},
       "fps": FPS, "max_frames": MAX_FRAMES, "min_len_s": MIN_LEN_S, "low_conf": LOW_CONF,
       "vocab": vocab, "temperature": temps, "trained_tasks": TRAIN_TASKS, "fewshot_tasks": [HOLDOUT],
       "smoke_test": SMOKE, "created": time.strftime("%Y-%m-%d %H:%M")}
json.dump(cfg, open(f"{BUNDLE}/config.json", "w"), indent=1)
for f in ["kd_model.py", "kd_metrics.py", "results.md", "results.json"]:
    shutil.copy(f, BUNDLE)
shutil.make_archive(f"{WORK}/kaushaldub_model", "zip", BUNDLE)
print("bundle:", sorted(os.listdir(BUNDLE)), f"| zip {os.path.getsize(BUNDLE + '.zip')/1e6:.1f} MB")

## 9. Final check: reload the bundle and run it the way the app will

In [ ]:
rec = StepRecognizer(BUNDLE, device="cpu")
print("tasks in bundle:", rec.tasks())
sample = next(v for v in VIDEOS if v["split"] == "test" and v["task"] == TARGET[0] and os.path.exists(feat_path("clip", v["id"])))
if SMOKE or not find_video(sample):
    out = rec.format(rec.predict_features(np.load(feat_path("clip", sample["id"])).astype(np.float32), sample["task"]), sample["task"])
else:
    out = rec.predict(find_video(sample), sample["task"])
print(json.dumps(out, indent=1)[:1500])
print("\nground truth:", [(s["label"], s["start"], s["end"]) for s in sample["segments"]])
# reloaded model must match the in-memory one
x = torch.from_numpy(np.load(feat_path("clip", sample["id"])).astype(np.float32))[None]
with torch.no_grad():
    a = MODELS["clip_tcn"].eval()(x, sample["task"])[-1]
    b = rec.model.cpu()(x, sample["task"])[-1]
assert torch.allclose(a, b, atol=1e-4), "reloaded weights differ"
print("bundle reload check passed")
if SMOKE:
    print("\nSMOKE TEST PASSED. Set SMOKE = False and Run All for the real run.")